### **01 — Análise exploratória**

**CRISP-DM fase 2.3 — exploração dos dados** · Telco Customer Churn
---

Este notebook percorre o **roteiro de oito etapas** da análise exploratória sobre a
base bruta de Telco Customer, usando o dicionário de dados montado em
[`0-dicionario-dados.ipynb`](0-dicionario-dados.ipynb) como mapa de tipos:

1. **Distribuição dos atributos** — o comportamento de cada característica analisada.
2. **Análise univariada** — distribuição de frequências e medidas de localização,
   dispersão, assimetria e curtose, uma variável por vez.
3. **Análise bivariada** — se a variação de uma característica é acompanhada pela
   variação de outra.
4. **Análise multivariada** — padrões e relações entre várias variáveis ao mesmo tempo.
5. **Valores ausentes** — onde falta dado e quanto essa falta pesa.
6. **Valores discrepantes** — registros cujo comportamento foge do esperado.
7. **Correlação** — qualquer relação estatística entre duas ou mais variáveis.
8. **Seleção de características** — a combinação de variáveis relevante para o
   aprendizado que vem depois.

Cada etapa grava as figuras em `docs/imagens/eda/` e a tabela que originou cada
figura em `reports/tables/eda-exploratoria/` — a regra do projeto é que
**nenhuma figura existe sem a tabela que a produziu**.


# Análise exploratória

Nessa fase será realizado uma análise exploratória dos dados seguindo as etapas apresentadas a seguir:

<img src="../docs/imagens/eda-steps.png">

1. Na distribuição dos atributos é possível identificar o comportamento de cada uma das características analisadas
2. A análise estatística univariada analisa a distribuição e dispersão dos dados: a análise da distribuição de frequências e a análise das medidas de localização, dispersão, assimetria e curtose dos dados. 
3.  Na avaliação estatística da causalidade, convencionou-se descrever essas análises bivariadas como formas de investigar se a variação de uma característica pode ser explicada pela variação de outra medida. 
4.  Análise multivariada é uma ferramenta que encontra padrões e relações entre várias variáveis, permitindo prever efeitos e mudanças que uma variável terá sobre a outra. 
5.  Detecção de valores ausentes é uma tarefa comum na área de ciência de dados para mensurar os impactos dessa ausência no conjunto de dados em estudo. 
6.  A detecção de outliers consiste na identificação de padrões em dados com um comportamento diferente do esperado. Estes padrões são muitas vezes referidos como anomalias. 
7.  Em probabilidade e estatística, correlação, dependência ou associação é qualquer relação estatística entre duas ou mais variáveis. 
8.  Os métodos de seleção de características têm a missão de encontrar uma combinação adequada de características relevantes para o aprendizado de um conceito.

# Importações

In [62]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

# Importações
import prince
import textwrap
import circlify
import pandas as pd
import numpy as np
import seaborn as sns
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.collections import EllipseCollection
from tqdm.auto import tqdm
from scipy.spatial.distance import cdist
from scipy.stats import chi2_contingency
from scipy.stats.contingency import association
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from itertools import combinations

In [63]:
# Funções para aplicar as escalas
def carregar_escalas(caminho):
    escala = pd.read_csv(caminho,
                         sep=None, engine='python',   # detecta , ou ;
                         dtype={'coluna': str, 'nivel': str},
                         encoding='utf-8-sig')        # remove BOM, se houver
    escala.columns = escala.columns.str.strip().str.lower()

    esperadas = {'coluna', 'nivel', 'ordem'}
    faltando = esperadas - set(escala.columns)
    if faltando:
        raise ValueError(f'Colunas ausentes em {caminho}: {faltando}. '
                         f'Colunas encontradas: {escala.columns.tolist()}')

    escala = escala.sort_values(['coluna', 'ordem'])
    return escala.groupby('coluna')['nivel'].apply(list).to_dict()


def aplicar_escalas(df, escalas):
    for coluna, niveis in escalas.items():
        if coluna not in df.columns:
            print(f'Aviso: a coluna "{coluna}" está na escala, mas não existe no dataset')
            continue

        # Se a coluna for numérica (ex.: notas 1 a 5), converte os níveis para número
        if pd.api.types.is_numeric_dtype(df[coluna]):
            niveis = pd.to_numeric(pd.Series(niveis)).tolist()

        df[coluna] = pd.Categorical(df[coluna], categories=niveis, ordered=True)
    return df

In [64]:
# Carregando base de dados
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Carregando dicionário dos dados
dicionario = pd.read_csv('../data/processed/dicionario.csv', sep=',')

# Carregando escalas
#escalas = carregar_escalas(caminho='../data/processed/escala.csv')

# Removendo a linha customerID do dicionario pois ela não será utilizada
dicionario = dicionario[~dicionario['variavel'].astype(str).str.contains('customerID', case=False, na=False)]

In [68]:
dicionario.columns

Index(['variavel', 'descricao', 'valores', 'Tipo'], dtype='str')

In [69]:
# selecionando as mesmas colunas do dicionário de dados
df = df.loc[:,dicionario.variavel]

In [ ]:
#xpto

In [70]:
# Exibindo o Dicionário gerado no notebook anterior
# dicionario = dicionario[~dicionario['variavel'].astype(str).str.contains('custumerID', case=False, na=False)]
dicionario

,variavel,descricao,valores,Tipo
1,gender,Whether the customer is a male or a female,"['Female', 'Male']",Nominal
2,SeniorCitizen,Whether the customer is a senior citizen or no...,"[0, 1]",Discreta
3,Partner,Whether the customer has a partner or not (Yes...,"['Yes', 'No']",Nominal
4,Dependents,Whether the customer has dependents or not (Ye...,"['No', 'Yes']",Nominal
5,tenure,Number of months the customer has stayed with ...,"[0, 72]",Discreta
6,PhoneService,Whether the customer has a phone service or no...,"['No', 'Yes']",Nominal
7,MultipleLines,Whether the customer has multiple lines or not...,"['No phone service', 'Yes']",Nominal
8,InternetService,"Customers internet service provider (DSL, Fibe...","['DSL', 'No']",Nominal
9,OnlineSecurity,Whether the customer has online security or no...,"['No', 'No internet service']",Nominal
10,OnlineBackup,Whether the customer has online backup or not ...,"['Yes', 'No internet service']",Nominal


# Análises iniciais

In [71]:
# Visão geral do dataset, contagem de nulos e tipo de dados das colunas
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   str    
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   str    
 3   Dependents        7043 non-null   str    
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   str    
 6   MultipleLines     7043 non-null   str    
 7   InternetService   7043 non-null   str    
 8   OnlineSecurity    7043 non-null   str    
 9   OnlineBackup      7043 non-null   str    
 10  DeviceProtection  7043 non-null   str    
 11  TechSupport       7043 non-null   str    
 12  StreamingTV       7043 non-null   str    
 13  StreamingMovies   7043 non-null   str    
 14  Contract          7043 non-null   str    
 15  PaperlessBilling  7043 non-null   str    
 16  PaymentMethod     7043 non-null   str    
 17  Monthl

In [ ]:
# Ajustando a Coluna TotalCharges que está como str e deveria ser float
# Converte para numérico, transformando espaços vazios em NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Arredonda para 2 casas decimais
df['TotalCharges'] = df['TotalCharges'].round(2)

In [73]:
# Removendo a Coluna customerID pois ela não será utilizada
# df = df.drop(columns=['customerID'], inplace=True)

In [74]:
# Quantidade por tipo
dicionario.Tipo.value_counts()

Tipo
Nominal     16
Discreta     2
Contínua     2
Name: count, dtype: int64

In [75]:
# Estatísticas descritivas
df.describe()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7043.000000,7032.000000
mean,0.162147,32.371149,64.761692,2283.300441
std,0.368612,24.559481,30.090047,2266.771362
min,0.000000,0.000000,18.250000,18.800000
25%,0.000000,9.000000,35.500000,401.450000
50%,0.000000,29.000000,70.350000,1397.475000
75%,0.000000,55.000000,89.850000,3794.737500
max,1.000000,72.000000,118.750000,8684.800000


# Análise univariada

## Características nominais

In [76]:
#Obter uma lista das variáveis nominais
nominal_columns = (
    dicionario
    .query('Tipo == "Nominal"')
    .variavel
    .to_list()
)

In [77]:
nominal_columns

['gender',
 'Partner',
 'Dependents',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'Churn']

In [78]:
if nominal_columns != []:
    for column in df.columns:
        if (column in nominal_columns):
            plt.switch_backend('TkAgg')
            # Gerando gráfico
            ## Configurando parâmetros
            ax = sns.barplot(x=df[column].value_counts().values[:10],y=df[column].value_counts().index[:10], color='#183EFF',alpha=1)

            ## Configurando estilo e tamanho
            sns.set_style("whitegrid")
            ax.figure.set_size_inches(25, 8)
            # Personalizando valores absolutos e relativos
            total = float(df[column].value_counts().values.sum())
            for i in ax.containers:
                widths = [j.get_width() for j in i]
                total = sum(widths)
                ax.bar_label(i, label_type='edge', labels=[f"{width:.0f}\n{width/total:.2%}" for width in widths])

            ## Adicionando barra com valor médio
            ax.axvline(df[column].value_counts().values.mean(), color="#FFD000", clip_on=False,ls='dashed')

            ## Configurações de cores e barras    
            sns.despine(left=False)
            ax.axvline(0, color="k", clip_on=False)
            plt.xlabel('Quantidade de registros')
            plt.title('As categorias mais frequentes da característica '+ column)

            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/univariada-nominal-'+column+'.png',bbox_inches='tight')
            plt.close(fig)
            ##plt.show()

## Características contínuas

In [79]:
#Obter uma lista das variáveis contínuas
continuous_columns = (
    dicionario
    .query('Tipo == "Contínua"')
    .variavel
    .to_list()
)

In [80]:
#Variáveis continuas
continuous_columns

['MonthlyCharges', 'TotalCharges']

In [81]:
if continuous_columns != []:
    for column in df.columns:
        if (column in continuous_columns):
            plt.switch_backend('TkAgg')
            # Gerando gráfico de valores mínimo e máximo
            ## Configurando parâmetros
            ax = sns.barplot(x=['Mínimo','Máximo'],y=[df[column].min(),df[column].max()], color='#183EFF',alpha=1)
            valores = [df[column].min(),df[column].max()]

            ## Configurando estilo e tamanho
            sns.set_style("whitegrid")
            ax.figure.set_size_inches(10, 8)

            # Adicionando valores sobre as barras
            ax.bar_label(container=ax.containers[0], labels=valores)

            # Adicionando barra com valor da diferença entre o mínimo e máximo
            ax.axhline(df[column].max() - df[column].min(), color="#FFD000", clip_on=False,ls='dashed')

            # Corrigindo valores do eixo Y
            current_values = plt.gca().get_yticks()
            plt.gca().set_yticks(current_values, [f'{x:.0f}' for x in current_values])

            ## Configurações de cores e barras    
            sns.despine(left=False)
            ax.axhline(0, color="k", clip_on=False)
            plt.ylabel(column)
            plt.title('Valores mínimo e máximo da característica '+ column)

            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/univariada-continua-'+column+'.png',bbox_inches='tight')
            plt.close()
            

            # Gerando gráfico de dispersão
            ## Configurando parâmetros
            ax = sns.scatterplot(data=df,x=column,y=column,color='#183EFF')

            ## Configurando estilo e tamanho
            sns.set_style("whitegrid")
            ax.figure.set_size_inches(10, 8)

            # Corrigindo valores do eixo x
            current_values = plt.gca().get_xticks()
            plt.gca().set_xticks(current_values, [f'{x:.0f}' for x in current_values])

            # Corrigindo valores do eixo y
            current_values = plt.gca().get_yticks()
            plt.gca().set_yticks(current_values, [f'{x:.0f}' for x in current_values])

            ## Configurações de cores e barras    
            #sns.despine(left=False,top=False)
            plt.title('Gráfico de dispersão da característica '+ column)
            plt.xlabel(column)

            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/univariada-continua-'+column+'-dispersao.png',bbox_inches='tight')        
            plt.close()

            # Gerando gráfico de caixa
            ## Configurando parâmetros
            ax = sns.boxplot(data=df,x=column, color='#183EFF',medianprops={"color": "#FFD000"})

            ## Configurando estilo e tamanho
            sns.set_style("whitegrid")
            ax.figure.set_size_inches(10, 8)

            # Corrigindo valores do eixo Y
            current_values = plt.gca().get_xticks()
            plt.gca().set_xticks(current_values, [f'{x:.0f}' for x in current_values])

            ## Configurações de cores e barras    
            #sns.despine(left=False,top=False)
            plt.title('Gráfico de caixa da característica '+ column)

            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/univariada-continua-'+column+'-caixa.png',bbox_inches='tight')
            plt.close()

            # Gráfico de caixa quartiles
            ## Calculando cada quartil da coluna
            q1 = df[column].quantile(0.25)
            q2 = df[column].quantile(0.5)
            q3 = df[column].quantile(0.75)

            ## Configurando parâmetros
            ax = sns.boxplot(x=[df[column].min(),q1,q2,q3], color='#183EFF',medianprops={"color": "#FFD000"})


            ## Configurando estilo e tamanho
            sns.set_style("whitegrid")
            ax.figure.set_size_inches(10, 8)

            # Corrigindo valores do eixo x
            current_values = plt.gca().get_xticks()
            plt.gca().set_xticks(current_values, [f'{x:.0f}' for x in current_values])

            ## Configurações de cores e barras    
            plt.title('Gráfico de caixa quartil da característica '+ column)
            plt.xlabel(column)

            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/univariada-continua-'+column+'caixa-quartil.png',bbox_inches='tight')        
            plt.close()

## Características discretas

In [82]:
#Obter uma lista das variáveis discretas
discrete_columns = (
    dicionario
    .query('Tipo == "Discreta"')
    .variavel
    .to_list()
)

In [83]:
discrete_columns

['SeniorCitizen', 'tenure']

In [84]:
if discrete_columns != []:
    for column in df.columns:
        if (column in discrete_columns):

            ## Configurando estilo e tamanho
            sns.set_style("whitegrid")
            #ax.figure.set_size_inches(10, 8)
            padrao = (10, 8)
            fig, ax = plt.subplots(figsize=padrao)

            # Gerando gráfico de valores mínimo e máximo
            ## Configurando parâmetros
            ax = sns.barplot(x=['Mínimo','Máximo'],y=[df[column].min(),df[column].max()], color='#183EFF',alpha=1)
            valores = [df[column].min(),df[column].max()]

            # Adicionando valores sobre as barras
            #ax.bar_label(container=ax.containers[0], labels=valores)
            for bars in ax.containers:
                ax.bar_label(bars)

            # Adicionando barra com valor da diferença entre o mínimo e máximo
            ax.axhline(df[column].max() - df[column].min(), color="#FFD000", clip_on=False,ls='dashed')

            # Corrigindo valores do eixo Y
            current_values = plt.gca().get_yticks()
            plt.gca().set_yticks(current_values, [f'{x:.0f}' for x in current_values])

            ## Configurações de cores e barras    
            sns.despine(left=False)
            ax.axhline(0, color="k", clip_on=False)
            plt.ylabel(column)
            plt.title('Valores mínimo e máximo da característica '+ column)

            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/univariada-discreta-'+column+'.png',bbox_inches='tight')
            plt.close()


            ## Configurando estilo e tamanho
            sns.set_style("whitegrid")
            #ax.figure.set_size_inches(10, 8)
            padrao = (10, 8)
            fig, ax = plt.subplots(figsize=padrao) 

            # Gerando gráfico de dispersão
            ## Configurando parâmetros
            ax = sns.scatterplot(data=df,x=column,y=column,color='#183EFF')


            # Corrigindo valores do eixo x
            current_values = plt.gca().get_xticks()
            plt.gca().set_xticks(current_values, [f'{x:.0f}' for x in current_values])

            # Corrigindo valores do eixo y
            current_values = plt.gca().get_yticks()
            plt.gca().set_yticks(current_values, [f'{x:.0f}' for x in current_values])

            ## Configurações de cores e barras    
            #sns.despine(left=False,top=False)
            plt.title('Gráfico de dispersão da característica '+ column)
            plt.xlabel(column)

            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/univariada-discreta-'+column+'-dispersao.png',bbox_inches='tight') 
            plt.close(fig)

            ## Configurando estilo e tamanho
            sns.set_style("whitegrid")
            #ax.figure.set_size_inches(10, 8)
            padrao = (10, 8)
            fig, ax = plt.subplots(figsize=padrao) 

            # Gerando gráfico de caixa
            ## Configurando parâmetros
            ax = sns.boxplot(data=df,x=column, color='#183EFF',medianprops={"color": "#FFD000"})


            # Corrigindo valores do eixo Y
            current_values = plt.gca().get_xticks()
            plt.gca().set_xticks(current_values, [f'{x:.0f}' for x in current_values])

            ## Configurações de cores e barras    
            #sns.despine(left=False,top=False)
            plt.title('Gráfico de caixa da característica '+ column)

            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/univariada-discreta-'+column+'-caixa.png',bbox_inches='tight')
            plt.close(fig)


            ## Configurando estilo e tamanho
            sns.set_style("whitegrid")
            #ax.figure.set_size_inches(10, 8)
            padrao = (10, 8)
            fig, ax = plt.subplots(figsize=padrao)

            # Gráfico de caixa quartiles
            ## Calculando cada quartil da coluna
            q1 = df[column].quantile(0.25)
            q2 = df[column].quantile(0.5)
            q3 = df[column].quantile(0.75)

            ## Configurando parâmetros
            ax = sns.boxplot(x=[df[column].min(),q1,q2,q3], color='#183EFF',medianprops={"color": "#FFD000"})


            # Corrigindo valores do eixo x
            current_values = plt.gca().get_xticks()
            plt.gca().set_xticks(current_values, [f'{x:.0f}' for x in current_values])

            ## Configurações de cores e barras    
            plt.title('Gráfico de caixa quartil da característica '+ column)
            plt.xlabel(column)

            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/univariada-discreta-'+column+'-caixa-quartil.png',bbox_inches='tight')  
            plt.close(fig)

## Características temporais

In [85]:
# Obter uma lista das variáveis de tempo
data_columns = (
    dicionario
    .query('Tipo == "Temporal"')
    .variavel
    .to_list()
)

In [86]:
# Não temos varáveis temporais no dataset
# data_columns = ['ano_construcao']

In [87]:
if data_columns != []:
    for column in df.columns:
        if (column in data_columns):

            # Contabilizando quantos registros no ano
            day_data_freq = df[column].value_counts().reset_index()
            day_data_freq.columns = ['data','qnt']

            # Convertendo object em data
            day_data_freq['data'] = pd.to_datetime(day_data_freq['data'], format='%Y') 

            # Gerando gráfico com linha do tempo
            ## Configurando parâmetros
            # ax.figure.set_size_inches(25, 8)
            padrao = (25, 8)
            fig, ax = plt.subplots(figsize=padrao)
            sns.set_style("whitegrid")
            
            ax = sns.lineplot(data=day_data_freq,x='data',y='qnt', palette="tab10", linewidth=2.5, color='#183EFF',alpha=.8)

            ## Adicionando barra com valor médio
            ax.axhline(day_data_freq['qnt'].mean(), color="#FFD000", clip_on=False,ls='dashed')

            ## Configurações de cores e barras    
            #sns.despine(left=False)
            ax.set(frame_on=False)
            #ax.axhline(0, color="k", clip_on=False)
            plt.ylabel('Quantidade de registros')
            plt.xlabel(column)
            plt.title('Distribuição na linha do tempo da característica '+ column)

            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/univariada-temporal-'+column+'.png',bbox_inches='tight')
            plt.close()

## Características ordinais

In [88]:
#Obter uma lista das variáveis de tempo
ordinal_columns = (
    dicionario
    .query('Tipo == "Ordinal"')
    .variavel
    .to_list()
)

In [89]:
#Não temos variáveis Ordinais
# ordinal_columns = ['qualidade_geral']

In [90]:
if ordinal_columns != []:
    for column in ordinal_columns:
        ## Configurando estilo e tamanho
        sns.set_style("whitegrid")
        #ax.figure.set_size_inches(10, 8)
        padrao = (10, 8)
        fig, ax = plt.subplots(figsize=padrao)

        # Gerando gráfico de valores mínimo e máximo
        ## Configurando parâmetros
        ax = sns.barplot(x=['Mínimo','Máximo'],y=[df[column].min(),df[column].max()], color='#183EFF',alpha=1)
        valores = [df[column].min(),df[column].max()]

        # Adicionando valores sobre as barras
        #ax.bar_label(container=ax.containers[0], labels=valores)
        for bars in ax.containers:
            ax.bar_label(bars)

        # Adicionando barra com valor da diferença entre o mínimo e máximo
        ax.axhline(df[column].max() - df[column].min(), color="#FFD000", clip_on=False,ls='dashed')

        # Corrigindo valores do eixo Y
        current_values = plt.gca().get_yticks()
        plt.gca().set_yticks(current_values, [f'{x:.0f}' for x in current_values])

        ## Configurações de cores e barras    
        sns.despine(left=False)
        ax.axhline(0, color="k", clip_on=False)
        plt.ylabel(column)
        plt.title('Valores mínimo e máximo da característica '+ column)

        ## Salvando figura
        fig = ax.get_figure()
        fig.savefig('../docs/imagens/eda/univariada-discreta-'+column+'.png',bbox_inches='tight')
        plt.close()


        ## Configurando estilo e tamanho
        sns.set_style("whitegrid")
        #ax.figure.set_size_inches(10, 8)
        padrao = (10, 8)
        fig, ax = plt.subplots(figsize=padrao) 

        # Gerando gráfico de dispersão
        ## Configurando parâmetros
        ax = sns.scatterplot(data=df,x=column,y=column,color='#183EFF')


        # Corrigindo valores do eixo x
        current_values = plt.gca().get_xticks()
        plt.gca().set_xticks(current_values, [f'{x:.0f}' for x in current_values])

        # Corrigindo valores do eixo y
        current_values = plt.gca().get_yticks()
        plt.gca().set_yticks(current_values, [f'{x:.0f}' for x in current_values])

        ## Configurações de cores e barras    
        #sns.despine(left=False,top=False)
        plt.title('Gráfico de dispersão da característica '+ column)
        plt.xlabel(column)

        ## Salvando figura
        fig = ax.get_figure()
        fig.savefig('../docs/imagens/eda/univariada-discreta-'+column+'-dispersao.png',bbox_inches='tight') 
        plt.close(fig)

        ## Configurando estilo e tamanho
        sns.set_style("whitegrid")
        #ax.figure.set_size_inches(10, 8)
        padrao = (10, 8)
        fig, ax = plt.subplots(figsize=padrao) 

        # Gerando gráfico de caixa
        ## Configurando parâmetros
        ax = sns.boxplot(data=df,x=column, color='#183EFF',medianprops={"color": "#FFD000"})


        # Corrigindo valores do eixo Y
        current_values = plt.gca().get_xticks()
        plt.gca().set_xticks(current_values, [f'{x:.0f}' for x in current_values])

        ## Configurações de cores e barras    
        #sns.despine(left=False,top=False)
        plt.title('Gráfico de caixa da característica '+ column)

        ## Salvando figura
        fig = ax.get_figure()
        fig.savefig('../docs/imagens/eda/univariada-discreta-'+column+'-caixa.png',bbox_inches='tight')
        plt.close(fig)


        ## Configurando estilo e tamanho
        sns.set_style("whitegrid")
        #ax.figure.set_size_inches(10, 8)
        padrao = (10, 8)
        fig, ax = plt.subplots(figsize=padrao)

        # Gráfico de caixa quartiles
        ## Calculando cada quartil da coluna
        q1 = df[column].quantile(0.25)
        q2 = df[column].quantile(0.5)
        q3 = df[column].quantile(0.75)

        ## Configurando parâmetros
        ax = sns.boxplot(x=[df[column].min(),q1,q2,q3], color='#183EFF',medianprops={"color": "#FFD000"})


        # Corrigindo valores do eixo x
        current_values = plt.gca().get_xticks()
        plt.gca().set_xticks(current_values, [f'{x:.0f}' for x in current_values])

        ## Configurações de cores e barras    
        plt.title('Gráfico de caixa quartil da característica '+ column)
        plt.xlabel(column)

        ## Salvando figura
        fig = ax.get_figure()
        fig.savefig('../docs/imagens/eda/univariada-discreta-'+column+'-caixa-quartil.png',bbox_inches='tight')  
        plt.close(fig)

### Análise Bivariada

### Nominal X Contínuas

In [91]:
if (continuous_columns != []) and (nominal_columns != []):
    for column1 in nominal_columns:
        for column2 in continuous_columns:
            bivariada = df.groupby(column1)[column2].mean().reset_index()
            bivariada.sort_values(by=column2,inplace=True,ascending=False)
            # Gerando gráfico
            ## Configurando parâmetros
            ax = sns.barplot(x=bivariada[column2][:10],y=bivariada[column1][:10], color='#183EFF',alpha=1)

            ## Configurando estilo e tamanho
            sns.set_style("whitegrid")
            ax.figure.set_size_inches(25, 8)

            # Personalizando valores absolutos e relativos
            total = float(bivariada[column2].mean())
            for i in ax.containers:
                widths = [j.get_width() for j in i]
                total = sum(widths)
                ax.bar_label(i, label_type='edge', labels=[f"{width:.0f}\n{width/total:.2%}" for width in widths])

            ## Adicionando barra com valor médio
            ax.axvline(bivariada[column2].mean(), color="#FFD000", clip_on=False,ls='dashed')

            # Corrigindo valores do eixo x
            current_values = plt.gca().get_xticks()
            plt.gca().set_xticks(current_values, [f'{x:.0f}' for x in current_values])

            ## Configurações de cores e barras    
            sns.despine(left=False)
            ax.axvline(0, color="k", clip_on=False)
            plt.xlabel(column2)
            plt.ylabel(column1)
            plt.title('Análise bivariada da característica '+ column1 +' com a média da característica ' + column2)

            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/bivariada-'+column2+' X '+column1+'.png',bbox_inches='tight')    
            plt.close()

## Nominal x Discretas

In [92]:
if (nominal_columns != []) and (discrete_columns != []):
    for column1 in nominal_columns:
        for column2 in discrete_columns:
            bivariada = df.groupby(column1)[column2].mean().reset_index()
            bivariada.sort_values(by=column2,inplace=True,ascending=False)
            # Gerando gráfico
            ## Configurando parâmetros
            sns.set_style("whitegrid")
            padrao = (25, 8)
            fig, ax = plt.subplots(figsize=padrao)
            ax = sns.barplot(x=bivariada[column2][:10],y=bivariada[column1][:10], color='#183EFF',alpha=1)


            # Personalizando valores absolutos e relativos
            total = float(bivariada[column2].mean())
            for i in ax.containers:
                widths = [j.get_width() for j in i]
                total = sum(widths)
                ax.bar_label(i, label_type='edge', labels=[f"{width:.0f}\n{width/total:.2%}" for width in widths])

            ## Adicionando barra com valor médio
            ax.axvline(bivariada[column2].mean(), color="#FFD000", clip_on=False,ls='dashed')

            # Corrigindo valores do eixo x
            current_values = plt.gca().get_xticks()
            plt.gca().set_xticks(current_values, [f'{x:.0f}' for x in current_values])

            ## Configurações de cores e barras    
            sns.despine(left=False)
            ax.axvline(0, color="k", clip_on=False)
            plt.xlabel(column2)
            plt.ylabel(column1)
            plt.title('Análise Bivariada da característica '+ column1 +' com a média da característica ' + column2)

            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/bivariada-'+column2+' X '+column1+'.png',bbox_inches='tight')   
            plt.close()

## Nominal x Tempo

In [93]:
pallette = [
'#00F537', 
'#FFFF00',
'#00C8FF',
'#FF6464',
'#00D000',
'#FFD000',
'#183EFF',
'#3C3C3C',
'#FF0000',
'#009600',
'#FF7D00',
'#00008C'
]

In [94]:
if (nominal_columns != []) and (data_columns != []):
    for time in data_columns:
        for nominal in nominal_columns:
            # Identificando as categorias mais frequentes
            grouped = df.groupby([nominal])[nominal].count().reset_index(name='Contagem').sort_values(by='Contagem',ascending=False)[:12]
            grouped = grouped[nominal].tolist()
            
            # Agrupando as categorias pelo tempo
            grouped_category = df.groupby([time,nominal])[nominal].count().reset_index(name='Contagem')
            
            # Selecionando as categorias mais frequentes
            grouped_category = grouped_category[grouped_category[nominal].isin(grouped)]

            # Convertendo coluna do tempo em datetime
            grouped_category[time] = pd.to_datetime(grouped_category[time], format='%Y')
            
            # Gerando gráfico com linha do tempo
            ## Configurando parâmetros
            sns.set_style("whitegrid")
            padrao = (25, 8)
            fig, ax = plt.subplots(figsize=padrao)
            
            ax = sns.lineplot(data=grouped_category,
                              x=time,
                              y='Contagem', 
                              hue=nominal,
                              palette=pallette[0:len(grouped)], 
                              linewidth=2.5, 
                              color='#183EFF',
                              alpha=1)

            ## Adicionando barra com valor médio
            ax.axhline(grouped_category['Contagem'].mean(), color="#FFD000", clip_on=False,ls='dashed')

            ## Configurações de cores e barras    
            #sns.despine(left=False)
            ax.set(frame_on=False)
            #ax.axhline(0, color="k", clip_on=False)
            plt.ylabel('Quantidade de registros')
            plt.xlabel(time)
            plt.title('Distribuição na linha do tempo da característica '+ time + ' x ' + nominal)
            
            ## Salvando figura
            fig = ax.get_figure()
            fig.savefig('../docs/imagens/eda/bivariada-'+nominal+' X '+time+'.png',bbox_inches='tight')   
            plt.close()

## Nominal x Ordinal

In [95]:
# True = percentual por linha (compara a distribuição entre categorias nominais)
# False = contagem absoluta
normalizar = True

# Garantindo que as variáveis são listas, e não strings
if isinstance(nominal_columns, str):
    nominal_columns = [nominal_columns]
if isinstance(ordinal_columns, str):
    ordinal_columns = [ordinal_columns]

if (nominal_columns != []) and (ordinal_columns != []):
    for ordinal in ordinal_columns:
        # A ordem dos níveis vem da escala aplicada no carregamento (aplicar_escalas)
        if not (isinstance(df[ordinal].dtype, pd.CategoricalDtype) and df[ordinal].cat.ordered):
            print(f'Aviso: "{ordinal}" não tem escala aplicada; coluna ignorada. '
                  f'Confira o escala.csv e rode aplicar_escalas antes deste script.')
            continue
        ordem = list(df[ordinal].cat.categories)

        for nominal in nominal_columns:
            if nominal == ordinal:
                continue

            # Identificando as 12 categorias nominais mais frequentes
            top = df[nominal].value_counts().head(12).index.tolist()
            subset = df[df[nominal].isin(top)]

            # Tabela de contingência (linhas = nominal, colunas = ordinal na ordem da escala)
            tabela = pd.crosstab(subset[nominal], subset[ordinal])
            tabela = tabela.reindex(index=top, columns=ordem, fill_value=0)

            if normalizar:
                dados = tabela.div(tabela.sum(axis=1), axis=0) * 100
                fmt = '.1f'
                legenda = '% dos registros da categoria'
            else:
                dados = tabela
                fmt = 'd'
                legenda = 'Quantidade de registros'

            # Gerando gráfico
            ## Configurando parâmetros
            sns.set_style("white")
            largura = max(8, 1.2 * len(ordem) + 4)
            altura = max(5, 0.6 * len(top) + 2)
            fig, ax = plt.subplots(figsize=(largura, altura))

            sns.heatmap(dados,
                        annot=True,
                        fmt=fmt,
                        cmap='Blues',
                        linewidths=0.5,
                        linecolor='white',
                        cbar_kws={'label': legenda},
                        ax=ax)

            ## Configurações de rótulos (quebra rótulos longos, ex.: 'Fusíveis de 60 A, fiação antiga')
            ax.set_xticklabels([textwrap.fill(str(n), 14) for n in ordem], rotation=0)
            plt.ylabel(nominal)
            plt.xlabel(ordinal)
            plt.yticks(rotation=0)
            plt.title('Tabela de contingência da característica ' + nominal + ' x ' + ordinal)

            ## Salvando figura
            fig.savefig('../docs/imagens/eda/bivariada-' + nominal + ' X ' + ordinal + '.png',
                        bbox_inches='tight')
            plt.close(fig)

# Análise Multivariada

In [96]:
"""
nominal_columns = [   
    'gender',
    'Partner',
    'Dependents',
    'PhoneService',
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'Contract',
    'PaperlessBilling',
    'PaymentMethod', 
    'Churn'   
]
"""
nominal_columns

['gender',
 'Partner',
 'Dependents',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'Churn']

In [97]:
ordinal_columns

[]

In [98]:
#: Nominais com cardinalidade que sustenta uma tabela de contingência estável.
COLUMNS_ACC = nominal_columns #+ ordinal_columns

#: Inércia acumulada mínima nas duas dimensões para que o plano seja interpretável.
INERCIA_MINIMA = 80.0

def distancia_euclidiana(px, qx, py, qy):
    """Distância entre dois pontos do plano fatorial."""
    return float(np.sqrt((px - qx) ** 2 + (py - qy) ** 2))


def circulos_do_agrupamento(vinculos: pd.DataFrame) -> list:
    """Hierarquia total → grupo → categoria, no formato que o circlify espera."""
    dados = [
        {
            "id": "Total",
            "datum": len(vinculos),
            "children": [
                {
                    "id": grupo,
                    "datum": int((vinculos.grupo == grupo).sum()),
                    "children": [
                        {"id": categoria, "datum": 1}
                        for categoria in vinculos.query("grupo == @grupo").categoria
                    ],
                }
                for grupo in vinculos.grupo.unique()
            ],
        }
    ]
    return circlify.circlify(
        dados, show_enclosure=False, target_enclosure=circlify.Circle(x=0, y=0, r=1)
    )


inercias = []

for p in range(len(COLUMNS_ACC)):
    for k in range(p + 1, len(COLUMNS_ACC)):
        primeira, segunda = COLUMNS_ACC[p], COLUMNS_ACC[k]

        # Tabela de contingência, sempre com a variável de maior cardinalidade nas linhas
        contingencia = pd.crosstab(df[primeira], df[segunda])
        if contingencia.shape[0] < contingencia.shape[1]:
            contingencia = contingencia.T
        # Remove categorias sem registros (níveis da escala que não aparecem nos dados)
        contingencia = contingencia.loc[contingencia.sum(axis=1) > 0, contingencia.sum(axis=0) > 0]

        # Com 2 categorias no menor eixo a CA tem só 1 dimensão: não há plano para desenhar
        if min(contingencia.shape) < 3:
            continue

        modelo = prince.CA(n_components=2, random_state=42).fit(contingencia)
        inercia = float(np.sum(modelo.percentage_of_variance_))
        inercias.append(
            {
                "linhas": contingencia.index.name,
                "colunas": contingencia.columns.name,
                "inercia_2d": round(inercia, 2),
                # Com 3 categorias no menor eixo a CA tem só 2 dimensões: 100% por construção
                "trivial": min(contingencia.shape) == 3,
                "figura": inercia >= INERCIA_MINIMA,
            }
        )
        if inercia < INERCIA_MINIMA:
            continue

        # Cada categoria de linha vai para a categoria de coluna mais próxima no plano
        linhas = modelo.row_coordinates(contingencia)
        colunas = modelo.column_coordinates(contingencia)
        pares = []
        for i, categoria in enumerate(contingencia.index):
            distancias = [
                distancia_euclidiana(
                    linhas.iloc[i, 0], colunas.iloc[j, 0], linhas.iloc[i, 1], colunas.iloc[j, 1]
                )
                for j in range(len(colunas))
            ]
            pares.append(
                {
                    "categoria": categoria,
                    "grupo": contingencia.columns[int(np.argmin(distancias))],
                    "distancia": round(min(distancias), 3),
                }
            )
        vinculos = pd.DataFrame(pares)
        if vinculos.grupo.nunique() < 2:
            continue

        circulos = circulos_do_agrupamento(vinculos)

        fig, ax = plt.subplots(figsize=(11, 11))
        ax.axis("off")
        ax.set_aspect("equal")
        limite = max(max(abs(c.x) + c.r, abs(c.y) + c.r) for c in circulos)
        ax.set_xlim(-limite, limite)
        ax.set_ylim(-limite, limite)

        for circulo in circulos:  # o círculo de cada grupo
            if circulo.level == 2:
                ax.add_patch(plt.Circle((circulo.x, circulo.y), circulo.r, color="#2A6F97"))

        for circulo in circulos:  # as categorias dentro de cada grupo
            if circulo.level == 3:
                ax.add_patch(
                    plt.Circle((circulo.x, circulo.y), circulo.r, alpha=0.55, color="#62B6CB")

                )
                ax.annotate(
                    circulo.ex["id"],
                    (circulo.x, circulo.y),
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="#12303F",
                )

        for circulo in circulos:  # as etiquetas dos grupos, por cima
            if circulo.level == 2:
                ax.annotate(
                    circulo.ex["id"],
                    (circulo.x, circulo.y + circulo.r * 0.82),
                    ha="center",
                    va="center",
                    fontsize=9,
                    color="white",
                    bbox=dict(
                        facecolor="#1B4965", edgecolor="none", boxstyle="round,pad=0.4"
                    ),
                )

        ax.set_title(
            f"{contingencia.index.name} agrupada por {contingencia.columns.name}"
            f" — inércia acumulada {inercia:.1f}%"
        )
        # salvar(fig, f"multivariada-ca-{primeira} X {segunda}", vinculos)
        ## Salvando figura
        fig.savefig('../docs/imagens/eda/acc-' + primeira + ' X ' + segunda + '.png', bbox_inches='tight')
        plt.close(fig)

# Salvando tabela de inércia dos pares
Path('../docs/tabelas').mkdir(parents=True, exist_ok=True)
inercias = pd.DataFrame(
    inercias, columns=["linhas", "colunas", "inercia_2d", "trivial", "figura"]
)
if inercias.empty:
    print("Nenhum par com >= 3 categorias em ambas as variáveis; nada para a CA.")
inercias = inercias.sort_values("inercia_2d", ascending=False)
inercias.to_csv('../docs/tabelas/multivariada-inercia-dos-pares.csv', index=False)


Versão 2.0 da Análise Multivariada. Essa Versão foi ajusta para o dataset atual

Adicionada proteções contra erros de processamento por alta cardinalidade de alguma coluna.

In [99]:
# ---------------------------------------------------------------- configuração
BASE_DOCS = Path("../docs")
DIR_IMG = BASE_DOCS / "imagens" / "eda"
DIR_TAB = BASE_DOCS / "tabelas"
DIR_IMG.mkdir(parents=True, exist_ok=True)
DIR_TAB.mkdir(parents=True, exist_ok=True)

SEED = 42

#: Acima disso a coluna não é tratada como categórica (IDs, textos livres...).
MAX_CARDINALIDADE = 20
#: Proteção: o circlify cresce ~n^3,5; acima disso a figura é pulada.
MAX_CATEGORIAS_FIGURA = 60
#: Troca "No internet service"/"No phone service" por "No" (redundância de codificação).
RECODIFICAR_ESTRUTURAIS = True

#: Critério de seleção dos pares para a CA. Com n ≈ 7 mil, quase todo teste dá
#: p < 0,05; quem filtra de verdade é o tamanho do efeito (V de Cramér).
ALFA = 0.05
V_MINIMO = 0.10
#: Inércia acumulada mínima no plano (só faz diferença em tabelas com ≥ 4 categorias
#: no menor eixo; com 3 categorias o plano tem 100% por construção).
INERCIA_MINIMA = 80.0

#: Variável-alvo: fica fora da MCA (segmentação não supervisionada) e serve só
#: para descrever os segmentos depois.
ALVO = "Churn"
N_DIM_MCA = 5
#: Dimensões da MCA com inércia ajustada abaixo disso (%) ficam fora do K-Means.
INERCIA_MIN_DIM = 1.0
K_CANDIDATOS = range(2, 9)

#### Seleção de colunas e recodificação estrutural

In [100]:
# Seleção de colunas e recodificação estrutural
ESTRUTURAIS = {
    "No internet service": "No",  # equivale a InternetService == "No"
    "No phone service": "No",     # equivale a PhoneService == "No"
}


def selecionar_colunas(dados: pd.DataFrame, colunas: list, max_card: int) -> list:
    """Mantém só colunas categóricas com cardinalidade analisável."""
    cardinalidade = dados[colunas].nunique()
    excluidas = cardinalidade[cardinalidade > max_card]
    if not excluidas.empty:
        print("Excluídas por cardinalidade:", excluidas.to_dict())
    return [c for c in colunas if cardinalidade[c] <= max_card]


def recodificar_estruturais(dados: pd.DataFrame, colunas: list) -> pd.DataFrame:
    """Remove a categoria que só repete a informação de InternetService/PhoneService."""
    dados = dados.copy()
    dados[colunas] = dados[colunas].replace(ESTRUTURAIS)
    return dados


COLUMNS_ACC = selecionar_colunas(df, nominal_columns, MAX_CARDINALIDADE)
df_cat = df[COLUMNS_ACC].astype(str)
if RECODIFICAR_ESTRUTURAIS:
    df_cat = recodificar_estruturais(df_cat, COLUMNS_ACC)

print(df_cat.nunique().sort_values(ascending=False).to_string())

PaymentMethod       4
InternetService     3
Contract            3
gender              2
PhoneService        2
MultipleLines       2
Dependents          2
Partner             2
OnlineBackup        2
OnlineSecurity      2
DeviceProtection    2
TechSupport         2
StreamingMovies     2
StreamingTV         2
PaperlessBilling    2
Churn               2


### Análise de valores ausentes

In [101]:
# Recarregando o dataset original para contabilizar a quantidade de registros por variável
df_original = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv', sep=',')

# Tipificando correntametne a coluna TotalCharges
df_original['TotalCharges'] = pd.to_numeric(df_original['TotalCharges'], errors='coerce')
# Arredonda para 2 casas decimais
df_original['TotalCharges'] = df_original['TotalCharges'].round(2)

counts = pd.DataFrame(df_original.count()).reset_index()
counts.columns = ['variavel', 'qnt']

In [102]:
## Configurando estilo e tamanho dos gráficos
sns.set_style("whitegrid")
padrao = (25, 12)
fig, ax = plt.subplots(figsize=padrao)

## Gerando gráfico de barras com a quantidade de registros por variável
ax = sns.barplot(data=counts, x='variavel', y='qnt', color='#183EFF', alpha=1)

# Personalizando valores absolutos
for i in ax.containers:
    ax.bar_label(i)

## Configurações de cores e barras
sns.despine(left=False)
ax.axhline(0, color="k", clip_on=False)
ax.tick_params(axis='x', rotation=45)
plt.xlabel('Características')
plt.ylabel('Quantidade de células')
plt.title('Análise do quantitativo de valores ausentes')

## Salvando figura
fig.savefig('../docs/imagens/eda/valores-ausentes.png', bbox_inches='tight', dpi=300)
plt.close(fig)

## Salvando tabela de valores ausentes
PASTA_TABELAS = Path('../docs/tabelas')
PASTA_TABELAS.mkdir(parents=True, exist_ok=True)

tabela_ausentes = counts[['variavel', 'qnt']].copy()
tabela_ausentes['percentual'] = (tabela_ausentes['qnt'] / len(df) * 100).round(2)
tabela_ausentes = tabela_ausentes.sort_values('qnt', ascending=False)
tabela_ausentes.to_csv(PASTA_TABELAS / 'valores-ausentes.csv', index=False, encoding='utf-8',sep=',')

### Detecção de valores discrepantes

In [103]:
# Criando função para calcular os limites de outlier pelo método IQR
def limites_IQR(data):
    """Retorna os limites abaixo/acima dos quais um valor é considerado outlier."""
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - 1.5 * IQR, Q3 + 1.5 * IQR


# Mapeando as colunas
outliers_columns = discrete_columns + continuous_columns

# Estrutura de repetição para mapear os outliers de cada variável
linhas = []
for column in outliers_columns:
    dados = df[column].dropna()
    limite_inferior, limite_superior = limites_IQR(dados)

    inferiores = dados[dados < limite_inferior]
    superiores = dados[dados > limite_superior]

    # Variáveis sem nenhum outlier ficam fora da tabela
    if inferiores.empty and superiores.empty:
        continue

    linhas.append({
        'variavel': column,
        'ponta_inferior': inferiores.min() if not inferiores.empty else None,
        'ponta_inferior_inicio': round(limite_inferior, 2) if not inferiores.empty else None,
        'ponta_superior_inicio': round(limite_superior, 2) if not superiores.empty else None,
        'ponta_superior': superiores.max() if not superiores.empty else None,
    })

outliers = pd.DataFrame(linhas)

# Renomeando as colunas para a leitura da área de negócio
outliers = outliers.rename(columns={
    'variavel': 'Característica',
    'ponta_inferior': 'Menor valor atípico',
    'ponta_inferior_inicio': 'Atípico abaixo de',
    'ponta_superior_inicio': 'Atípico acima de',
    'ponta_superior': 'Maior valor atípico',
})

# Salvando outliers
Path('../docs/tabelas').mkdir(parents=True, exist_ok=True)
outliers.to_csv('../docs/tabelas/outliers.csv', sep=',', index=False)
outliers

,Característica,Menor valor atípico,Atípico abaixo de,Atípico acima de,Maior valor atípico
0,SeniorCitizen,None,None,0.0,1


### Correlação de variáveis Numéricas

In [104]:
def correlograma_circulos(corr, titulo='Matriz de correlação', anotar=None,
                         cmap=None, figsize=None, ax=None):
    """Correlograma com círculos no triângulo inferior.
 
    A área de cada círculo é proporcional à força da correlação (|r|):
    círculo grande = forte, pequeno = fraca. A cor mostra o sinal e o valor,
    de -1 (azul) a 1 (vermelho).
    """
    n = len(corr)
    if anotar is None:
        anotar = n <= 15  # com muitas variáveis os números ficam ilegíveis
    if cmap is None:
        cmap = sns.diverging_palette(230, 20, as_cmap=True)
    if figsize is None:
        lado = max(6, 0.45 * n + 2)
        figsize = (lado + 1.5, lado)
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure
 
    valores = corr.to_numpy()
 
    # Triângulo inferior sem a diagonal (a diagonal é sempre 1)
    linhas, colunas = np.tril_indices(n, k=-1)
    r = valores[linhas, colunas]
 
    # Sem a primeira linha e a última coluna, que ficariam vazias
    x = colunas
    y = linhas - 1
 
    # Diâmetro pela raiz de |r|: assim a ÁREA do círculo é proporcional a |r|
    diametros = 0.9 * np.sqrt(np.abs(r))
 
    circulos = EllipseCollection(
        widths=diametros,
        heights=diametros,
        angles=np.zeros_like(r, dtype=float),
        units='x',
        offsets=np.column_stack([x, y]),
        offset_transform=ax.transData,
        cmap=cmap,
        edgecolor='white',
        linewidth=0.3,
    )
    circulos.set_array(r)
    circulos.set_clim(-1, 1)
    ax.add_collection(circulos)
 
    if anotar:
        for xi, yi, ri in zip(x, y, r):
            ax.text(xi, yi, f'{ri:.2f}'.replace('0.', '.', 1) if abs(ri) < 1 else f'{ri:.0f}',
                    ha='center', va='center', fontsize=7,
                    color='white' if abs(ri) > 0.6 else '#333333')  # círculos pequenos: texto escuro
 
    # Eixos e grade
    nomes = list(corr.columns)
    ax.set_xticks(range(n - 1))
    ax.set_xticklabels(nomes[:-1], rotation=90)
    ax.set_yticks(range(n - 1))
    ax.set_yticklabels(nomes[1:])
    ax.set_xlim(-0.5, n - 1.5)
    ax.set_ylim(n - 1.5, -0.5)
    ax.set_aspect('equal')
    ax.set_xticks(np.arange(-0.5, n - 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n - 1), minor=True)
    ax.grid(which='minor', color='#EEEEEE', linewidth=0.6)
    ax.tick_params(which='minor', length=0)
    for lado in ax.spines.values():
        lado.set_visible(False)
 
    barra = fig.colorbar(circulos, ax=ax, shrink=0.5, pad=0.02)
    barra.set_label('Correlação')
    barra.outline.set_visible(False)
 
    ax.set_title(titulo)
    return fig, ax
#xpto

In [105]:
corr = df[colunas].corr(method='pearson')
fig, ax = correlograma_circulos(corr, titulo='Correlação entre as variáveis numéricas')
fig.savefig('../docs/imagens/eda/correlacao.png', bbox_inches='tight', dpi=300)
plt.close(fig)

#### Correlação de Variáveis Categóricas
Inclui as binárias: elas não geram plano fatorial, mas a associação delas
também importa.

In [106]:
def associacao_par(x: pd.Series, y: pd.Series) -> dict:
    tabela = pd.crosstab(x, y)
    tabela = tabela.loc[tabela.sum(axis=1) > 0, tabela.sum(axis=0) > 0]
    if min(tabela.shape) < 2:
        return {"qui2": np.nan, "p_valor": np.nan, "v_cramer": np.nan}
    qui2, p, _, _ = chi2_contingency(tabela)
    return {
        "qui2": qui2,
        "p_valor": p,
        "v_cramer": association(tabela, method="cramer"),
    }


associacoes = pd.DataFrame(
    [
        {"var_1": a, "var_2": b, **associacao_par(df_cat[a], df_cat[b])}
        for a, b in combinations(COLUMNS_ACC, 2)
    ]
).sort_values("v_cramer", ascending=False)
associacoes.round(4).to_csv(DIR_TAB / "multivariada-associacao-dos-pares.csv", index=False)

# Matriz simétrica para o heatmap
matriz_v = (
    associacoes.pivot(index="var_1", columns="var_2", values="v_cramer")
    .reindex(index=COLUMNS_ACC, columns=COLUMNS_ACC)
)
# Espelha o triângulo e preenche a diagonal (só ela fica vazia depois do espelho)
matriz_v = (
    matriz_v.combine_first(matriz_v.T)
    .reindex(index=COLUMNS_ACC, columns=COLUMNS_ACC)
    .fillna(1.0)
)

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(matriz_v.values, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(COLUMNS_ACC)), COLUMNS_ACC, rotation=60, ha="right")
ax.set_yticks(range(len(COLUMNS_ACC)), COLUMNS_ACC)
for i in range(len(COLUMNS_ACC)):
    for j in range(len(COLUMNS_ACC)):
        valor = matriz_v.iat[i, j]
        ax.text(j, i, f"{valor:.2f}", ha="center", va="center", fontsize=7,
                color="white" if valor > 0.6 else "#12303F")
fig.colorbar(im, ax=ax, shrink=0.8, label="V de Cramér")
ax.set_title("Associação entre variáveis categóricas (V de Cramér)")
fig.savefig(DIR_IMG / "Correlacao_v_cramer.png", bbox_inches="tight", dpi=120)
#plt.show()

associacoes.head(10)

,var_1,var_2,qui2,p_valor,v_cramer
105,StreamingTV,StreamingMovies,1999.294942,0.000000e+00,0.533094
15,Partner,Dependents,1441.245221,0.000000e+00,0.452676
43,PhoneService,InternetService,1441.623387,0.000000e+00,0.452425
69,InternetService,StreamingTV,1382.051124,7.787742e-301,0.442979
70,InternetService,StreamingMovies,1381.710184,9.235186e-301,0.442924
116,Contract,Churn,1184.596572,5.863038e-258,0.410116
94,DeviceProtection,StreamingMovies,1137.066342,2.903000e-249,0.402111
65,InternetService,OnlineSecurity,1094.027917,2.721800e-238,0.394126
68,InternetService,TechSupport,1078.765077,5.612288e-235,0.391367
93,DeviceProtection,StreamingTV,1074.356205,1.237305e-235,0.390874


# Seleção de características

In [107]:
# Mantendo apenas as colunas que serão utilizadas no modelo
df_model = df.drop(columns=['Churn'])

# Salvando o dataset final para modelagem
df_model.to_csv('../data/processed/Telcon-Customer-Churn_columns_selected.csv', index=False, sep=',')

# Insights

In [109]:
insights = [
    "Só 26,5% dos clientes saíram (1.869 de 7.042), cerca de 1 em 4. A base não tem `customerID` "
    "duplicado. `Churn` não entra na MCA nem no K-Means e serve apenas para descrever os segmentos depois.",

    "A única coluna com ausência é `TotalCharges` (11 células, 0,16%), e todas têm `tenure` = 0: são "
    "clientes recém-contratados (10 de 11 em contrato de 2 anos) que ainda não foram cobrados. "
    "É ausência estrutural, então a correção é imputar 0, não excluir nem usar a mediana.",

    "O tipo de contrato é o que mais separa quem sai de quem fica (V de Cramér com `Churn` = 0,41). "
    "O mensal é 55% da base e tem 42,7% de churn, contra 11,3% no anual e 2,8% no de 2 anos. "
    "Sozinho, o contrato mensal responde por 88,6% de todos os cancelamentos.",

    "O churn está concentrado no início do relacionamento: 53% de quem tem até 6 meses sai, contra "
    "9,5% de quem passa de 48 meses. A mediana de `tenure` é 10 meses entre os que saem e 38 entre os "
    "que ficam. A distribuição tem dois polos (1.370 clientes com até 6 meses e 1.483 com mais de 60) "
    "e curtose de -1,39, ou seja, achatada.",

    "A fibra óptica é o maior foco de perda: 44% da base, 41,9% de churn (DSL 19,0%, sem internet 7,4%) "
    "e 69% de todos os cancelamentos. Também é o plano mais caro, com mensalidade média de 91,50 "
    "contra 58,10 no DSL.",

    "Segurança online e suporte técnico protegem o cliente: quem não tem esses serviços sai a "
    "41–42%, contra 15% de quem tem. O churn cai de 56,7% (nenhum serviço de proteção) para 5,3% "
    "(os quatro), o que sugere uma variável derivada de contagem de serviços. 57% dos clientes de "
    "fibra não têm segurança nem suporte, e esses saem a 55%.",

    "Quem paga por cheque eletrônico (33,6% da base) tem 45,3% de churn, contra 16,0% de quem usa "
    "pagamento automático (cartão ou débito, 43,5% da base). Os dois automáticos ficam no mesmo "
    "patamar (15,2% e 16,7%) e podem ser agrupados.",

    "A combinação contrato mensal + fibra + cheque eletrônico reúne 18,6% dos clientes, tem 60,4% de "
    "churn e concentra 42% de todas as saídas. É o candidato natural a cluster de alto risco, a ser "
    "confirmado na segmentação.",

    "`MonthlyCharges` é bimodal: 23% pagam até 30 (quem só tem telefone ou internet básica) e o resto "
    "se espalha até 118,75. O churn sobe com o valor: 10,9% até 35, 23,9% entre 35 e 70 e 37,8% entre "
    "70 e 90, estabilizando em 32,8% acima de 90. Quem sai paga em média 74,40, contra 61,30 de quem fica.",

    "`TotalCharges` é quase `tenure` × `MonthlyCharges` (razão mediana 1,00; correlação 0,83 com "
    "`tenure`). É a variável mais assimétrica da base (0,96) e é redundante. Na preparação, vale manter "
    "só uma delas ou criar a cobrança média mensal, para o K-Means não contar o tempo duas vezes.",

    "Há pares de categóricas quase redundantes: `StreamingTV` e `StreamingMovies` (V = 0,53), "
    "`Partner` e `Dependents` (0,45), `PhoneService` e `InternetService` (0,45). Os 6 serviços "
    "dependentes de internet carregam a categoria 'No internet service' (21,7%, V de 0,38–0,44 com "
    "`InternetService`) e `MultipleLines` carrega 'No phone service' (9,7%). A recodificação para 'No' "
    "evita contar a mesma informação duas vezes na MCA.",

    "`gender` não discrimina nada: 26,9% de churn entre mulheres e 26,2% entre homens. Já perfil "
    "familiar e idade discriminam: idosos saem a 41,7% (contra 23,6%), quem não tem parceiro a 33,0% "
    "(contra 19,7%) e quem não tem dependentes a 31,3% (contra 15,5%). `gender` pode ficar fora do "
    "modelo e os demais são bons descritores de segmento.",

    "`PaperlessBilling` (59% da base) parece forte, com 33,6% de churn contra 16,3%. Mas está associado "
    "a `InternetService` (V = 0,38), então boa parte do efeito é só o da fibra. Já `StreamingTV` e "
    "`StreamingMovies` têm churn próximo ao de quem não assina (30% contra 33,5%). A associação "
    "bivariada não deve ser lida como causa.",

    "O critério IQR sinaliza apenas `SeniorCitizen` (1.142 clientes, 16,2%), que é binária, então é "
    "falso positivo. `tenure`, `MonthlyCharges` e `TotalCharges` não têm nenhum outlier, e não há "
    "o que tratar nem remover nesta base.",
    
]

insights = pd.DataFrame(insights, columns=["insights"])

# Salvando insights
insights.to_csv('../docs/tabelas/insights.csv', index=False,sep=',')